In [10]:
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm


from sklearn.ensemble import HistGradientBoostingRegressor

from catboost import CatBoostRegressor
from xgboost import XGBRegressor
import lightgbm as lgb

import matplotlib.pyplot as plt
import seaborn as sns

In [11]:
ROOT_DIR: Path = Path.cwd().parent
DATA_DIR: Path = ROOT_DIR / Path("data")
OUT_DIR: Path = ROOT_DIR / Path("outputs") 

DATASET_DIR: Path = OUT_DIR / Path("datasets")

In [12]:
TRAIN_FEATURES_PATH: Path = DATASET_DIR / Path("train_features.csv")
TRAIN_TARGET_PATH: Path = DATASET_DIR / Path("train_target.csv")
TEST_FEATURES_PATH: Path = DATASET_DIR / Path("test_features.csv")
TEST_TARGET_PATH: Path = DATASET_DIR / Path("test_target.csv")

In [13]:
df_train_f: pd.DataFrame = pd.read_csv(TRAIN_FEATURES_PATH, parse_dates=["datetime"])
df_train_t: pd.DataFrame = pd.read_csv(TRAIN_TARGET_PATH, parse_dates=["datetime"])

df_test_f: pd.DataFrame = pd.read_csv(TEST_FEATURES_PATH, parse_dates=["datetime"])
df_test_t: pd.DataFrame = pd.read_csv(TEST_TARGET_PATH, parse_dates=["datetime"])

In [14]:
df_train_f.head(2)

,datetime,month,hour_of_day,wind_speed_10m,wind_speed_80m,wind_speed_120m,wind_speed_180m,wind_direction_10m,wind_direction_80m,wind_direction_120m,...,pressure_msl_minus_ps,temp80_minus_t2m,wd10_openm_minus_nasa_circular,ws10_openm_minus_nasa,ws80_openm_minus_nasa50,cf_empirical_iso_120,cf_empirical_iso_80,p_empirical_iso_120,p_empirical_iso_80,p_empirical_iso_mean_80_120
0,2022-01-01 00:00:00,1,0,2.77,4.27,4.62,NaN,0.244,0.249,0.252,...,915.54,2.16,-28.4,1.07,2.20,0.116720,0.122968,9.302024,9.799961,9.550993
1,2022-01-01 01:00:00,1,1,2.91,4.40,4.78,NaN,0.243,0.249,0.250,...,915.38,2.04,-16.0,0.81,1.81,0.128357,0.135182,10.229389,10.773326,10.501357


In [15]:
df_train_t.head(2)

,datetime,target
0,2022-01-01 00:00:00,0.697
1,2022-01-01 01:00:00,4.413


In [16]:
# Параметры ансамбля после Optuna

BLEND_WEIGHTS: dict[str: float] = {
    "cat_mae_direct": 0.361831,
    "xgb_residual": 0.177644,
    "lgb_residual": 0.064663,
    "hgb_q545": 0.235409,
    "hgb_q570": 0.116581,
    "hgb_q530": 0.043873,
}

ENSEMBLE_PARAMS: dict[str: float] = {
    "cat_iter": 1200,
    "xgb_estimators": 900,
    "lgb_estimators": 900,
    "hgb_iter": 650,
}

HGB_QUANTILES: list[int] = [0.545, 0.570, 0.530]

In [17]:
RANDOM_STATE: int = 42
INSTALLED_CAPACITY_MW: float = 90.09

feature_columns: list[str] = df_train_f.columns.drop("datetime").tolist()
x_train: pd.DataFrame = df_train_f[feature_columns]
y_train: pd.Series = df_train_t["target"]
p_empirical_train: pd.Series = df_train_f["p_empirical_mean_80_120"]

ensemble_models: dict[str, object] = {
    "cat_mae_direct": CatBoostRegressor(
        iterations=ENSEMBLE_PARAMS["cat_iter"],
        learning_rate=0.03,
        depth=6,
        loss_function="MAE",
        random_seed=RANDOM_STATE,
        verbose=0,
        allow_writing_files=False,
    ),
    "xgb_residual": XGBRegressor(
        objective="reg:absoluteerror",
        n_estimators=ENSEMBLE_PARAMS["xgb_estimators"],
        learning_rate=0.025,
        max_depth=5,
        min_child_weight=20,
        subsample=0.9,
        colsample_bytree=0.9,
        reg_lambda=2.0,
        tree_method="hist",
        random_state=RANDOM_STATE,
    ),
    "lgb_residual": lgb.LGBMRegressor(
        objective="regression_l1",
        n_estimators=ENSEMBLE_PARAMS["lgb_estimators"],
        learning_rate=0.025,
        num_leaves=31,
        min_child_samples=35,
        subsample=0.9,
        colsample_bytree=0.9,
        reg_lambda=1.0,
        random_state=RANDOM_STATE,
        verbosity=-1,
    ),
}

for quantile in HGB_QUANTILES:
    model_name: str = f"hgb_q{int(quantile * 1000):03d}"
    ensemble_models[model_name] = HistGradientBoostingRegressor(
        loss="quantile",
        quantile=quantile,
        max_iter=ENSEMBLE_PARAMS["hgb_iter"],
        learning_rate=0.04,
        max_leaf_nodes=31,
        min_samples_leaf=30,
        l2_regularization=0.02,
        early_stopping=True,
        validation_fraction=0.12,
        random_state=RANDOM_STATE,
    )

training_progress = tqdm(
    ensemble_models.items(),
    total=len(ensemble_models),
    desc="Обучение ансамбля",
    unit="модель",
)

for model_name, model in training_progress:
    training_progress.set_postfix(model=model_name)

    if model_name.endswith("residual"):
        model.fit(x_train, y_train - p_empirical_train)
    elif model_name.startswith("hgb"):
        model.fit(x_train, (y_train / INSTALLED_CAPACITY_MW).clip(0, 1))
    else:
        model.fit(x_train, y_train.clip(0, INSTALLED_CAPACITY_MW))

Обучение ансамбля: 100%|██████████| 6/6 [00:57<00:00,  9.59s/модель, model=hgb_q530]      


In [19]:
feature_importance: pd.DataFrame = pd.DataFrame({
    model_name: model.feature_importances_ / model.feature_importances_.sum()
    for model_name, model in ensemble_models.items()
    if hasattr(model, "feature_importances_")
}, index=feature_columns)

feature_importance["mean_importance"] = feature_importance.mean(axis=1)
feature_importance = feature_importance.sort_values("mean_importance", ascending=False).head(20)

feature_importance.round(4)

,cat_mae_direct,xgb_residual,lgb_residual,mean_importance
ws80_openm_minus_nasa50,0.0391,0.0276,0.0310,0.0326
p_empirical_minus_theory_120,0.0463,0.0113,0.0112,0.0229
dayofyear_cos,0.0189,0.0066,0.0339,0.0198
dayofyear_sin,0.0170,0.0062,0.0359,0.0197
wind_gusts_10m,0.0260,0.0065,0.0127,0.0151
ws10_openm_minus_nasa,0.0207,0.0052,0.0179,0.0146
temp80_minus_t2m,0.0125,0.0060,0.0234,0.0140
wind_speed_80m_nwp_plus_1h,0.0134,0.0091,0.0185,0.0137
wind_speed_120m_nwp_plus_2h,0.0237,0.0078,0.0091,0.0136
air_density,0.0096,0.0081,0.0224,0.0134


In [20]:
x_test: pd.DataFrame = df_test_f[feature_columns]
y_test: pd.Series = df_test_t["target"]
p_empirical_test: pd.Series = df_test_f["p_empirical_mean_80_120"]

model_predictions: dict[str, np.ndarray] = {}

for model_name, model in ensemble_models.items():
    if model_name.endswith("residual"):
        prediction: np.ndarray = p_empirical_test.to_numpy() + model.predict(x_test)
    elif model_name.startswith("hgb"):
        prediction: np.ndarray = model.predict(x_test) * INSTALLED_CAPACITY_MW
    else:
        prediction: np.ndarray = model.predict(x_test)

    model_predictions[model_name] = np.clip(prediction, 0, INSTALLED_CAPACITY_MW)

ensemble_prediction: np.ndarray = np.clip(
    sum(BLEND_WEIGHTS[model_name] * prediction for model_name, prediction in model_predictions.items()),
    0,
    INSTALLED_CAPACITY_MW,
)

test_error: float = np.mean(
    np.abs(y_test.to_numpy() - ensemble_prediction) / INSTALLED_CAPACITY_MW
) * 100

print(f"Ошибка ансамбля на test: {test_error:.4f}%")

Ошибка ансамбля на test: 7.9979%
